<a href="https://colab.research.google.com/github/gauravd12345/language_models/blob/main/rnn/rnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [233]:
import re
import random
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

# import nltk
# nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

device: cuda


In [234]:
""" Hyperparameters """

train_size = 0.8
seq_len = 10
lr = 0.001
epochs = 30
batch_size = 256
hidden_size = 300
embedding_dim = 300

In [235]:
with open("/content/tiny_shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

text = text.replace('\n', ' newline ')

text = word_tokenize(text)
print(text[:10])
vocab = sorted(set(text))

# vocab = "".join(sorted(set(text)))
vocab_len = len(vocab)

ctoi, itoc = {}, {}
for i in range(len(vocab)):
    ctoi[vocab[i]] = i
    itoc[i] = vocab[i]

print(f"vocab len: {vocab_len}")

['First', 'Citizen', ':', 'newline', 'Before', 'we', 'proceed', 'any', 'further', ',']
vocab len: 14211


In [236]:
X, y = [], []
for i in range(len(text) - seq_len - 1):
    x_i = [ctoi[word] for word in text[i : i + seq_len]]
    y_i = [ctoi[word] for word in text[i + 1 : i + seq_len + 1]]

    X.append(x_i)
    y.append(y_i)

X = torch.tensor(X, dtype=torch.long)
y = torch.tensor(y, dtype=torch.long)

""" train test split """
n = len(X)
train_end = int(train_size * n)

X_train = X[:train_end]
X_test = X[train_end:]

y_train = y[:train_end]
y_test = y[train_end:]

print(f"\nX_shape: {X.shape}")
print(f"Y_shape: {y.shape}")


X_shape: torch.Size([294759, 10])
Y_shape: torch.Size([294759, 10])


In [237]:
class RNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(vocab_len, embedding_dim)
        self.x_t = nn.Linear(embedding_dim, hidden_size) # W_e
        self.s_t = nn.Linear(hidden_size, hidden_size) # W_h
        self.y_t = nn.Linear(hidden_size, vocab_len)


    def forward(self, w, h):
        out = []
        e = self.embed(w) # shape is (batch_size, seq_len, embedding_dim)

        for i in range(seq_len):
          x = e[:, i, :] # column-wise word embeddings (batch_size, embedding_dim)

          x = self.x_t(x) + self.s_t(h) # (batch_size, hidden_size)

          h = torch.tanh(x)
          y = self.y_t(h)  # (batch_size, vocab_len)

          out.append(y)

        return torch.stack(out, dim=1), h

model = RNN().to(device)

In [238]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X_train, y_train)
loader = DataLoader(dataset, batch_size=batch_size, drop_last=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)
for epoch in range(epochs):
    total_loss = 0

    h = torch.ones(batch_size, hidden_size).to(device) * 0.1  # initial hidden state
    for xb, yb in loader:
        h = h.detach()  # clearing hidden state after each batch
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        out, h = model(xb, h)

        loss = criterion(out.reshape(-1, vocab_len), yb.reshape(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

Epoch 1, Loss: 5.2710
Epoch 2, Loss: 4.5407
Epoch 3, Loss: 4.1966
Epoch 4, Loss: 3.9374
Epoch 5, Loss: 3.7317
Epoch 6, Loss: 3.5620
Epoch 7, Loss: 3.4181
Epoch 8, Loss: 3.2940
Epoch 9, Loss: 3.1836
Epoch 10, Loss: 3.0835
Epoch 11, Loss: 2.9934
Epoch 12, Loss: 2.9131
Epoch 13, Loss: 2.8400
Epoch 14, Loss: 2.7731
Epoch 15, Loss: 2.7114
Epoch 16, Loss: 2.6536
Epoch 17, Loss: 2.5994
Epoch 18, Loss: 2.5495
Epoch 19, Loss: 2.5038
Epoch 20, Loss: 2.4609
Epoch 21, Loss: 2.4211
Epoch 22, Loss: 2.3838
Epoch 23, Loss: 2.3486
Epoch 24, Loss: 2.3161
Epoch 25, Loss: 2.2866
Epoch 26, Loss: 2.2580
Epoch 27, Loss: 2.2312
Epoch 28, Loss: 2.2059
Epoch 29, Loss: 2.1819
Epoch 30, Loss: 2.1593


In [260]:

start_seq = [ctoi[tok] for tok in text[:seq_len]]
data = torch.tensor(start_seq, dtype=torch.long, device=device).unsqueeze(0)  # (1, seq_len)

length = 200
model.eval()
h = torch.zeros(1, hidden_size, device=device)

for idx in start_seq:
    print(itoc[idx] if itoc[idx] != 'newline' else '\n', end=" ")

with torch.no_grad():
    for i in range(length):
        logits, h = model(data, h)  # logits: (1, seq_len, vocab_len)

        # take prediction from last timestep
        probs = torch.softmax(logits[0, -1, :], dim=0)
        pred = torch.multinomial(probs, num_samples=1).item()

        print(itoc[pred] if itoc[pred] != 'newline' else '\n', end=" ")

        data = torch.cat([data[:, 1:], torch.tensor([[pred]], device=device)], dim=1)

First Citizen : 
 Before we proceed any further , 
 Is your affairs , 
 To help your honour safe ! 
 
 DUKE VINCENTIO : 
 Love it had yet never wast behold ; 
 The baby beats my heart is pity must say , paying for some other reasons . What hither how does . 
 
 ANGELO : 
 Plainly conceive , but most likely ! 
 
 
 Second Gentleman : 
 Since , have to lose 
 At each his pity , how now ! 
 
 ANGELO : 
 Plainly conceive ? strike up the life of her , mark it , sentenced ; your trade . 
 Which I protest good my lord ? 
 
 ISABELLA : 
 Ignomy in ransom and free pardon . 
 
 ANGELO : 
 Plainly no money of my house . 
 
 ISABELLA : 
 With whom I would seek 
 As thou art always figuring diseases thou , a word . 
 
 ANGELO : 
 Plainly . 
 
 ISABELLA : 
 Sir , before life , most 
 
 Why , will in the office , 
 I do arrest you are : let these withdraw together , thou art , 